In [ ]:
#!/scratch/nnenv/bin/python3

%pip install watermark numpy pandas matplotlib torch torchvision torchaudio | grep -v 'already satisfied'
%load_ext watermark
%watermark -v -p watermark,numpy,pandas,matplotlib,torch

#
import math,random,os,sys,copy,time,glob,itertools,shutil,re, json
from scipy.stats import wilcoxon

#
import numpy as np

#
import torch, gc
torch.cuda.empty_cache()
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch.nn as nn
import torch.nn.functional as tf
from torch.utils.data import Dataset,DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor, Lambda

import pandas as pd

from pathlib import Path
sys.path[0] = str(Path(sys.path[0]).parent)

#
from demofcns import *
from opts.asgm.torchlasgm import PID
from opts.elpf.fo import LPF
from opts.elpf.esfo import esLPF

# from .. import arcs
from arcs.resnet import ResNetC

# python3 -m torch.utils.collect_env

##### Setup Configurations (Hyperparameters and so on)

In [ ]:
# - is it being run remotely on OSU EECS server?
remotelyrun = 1 # 0 | 1

#----------------------
# Load setup config
#----------------------
with open('cfgs.json', 'r') as cfglist:
  cfgs = json.load(cfglist)
  
# setup
cfgs['storedir'] = "../expstore"

# - pick data-set
cfgs["dataset"] = "cifar10"
# - set max. batchsize for each epoch, training runs, training epochs, 
cfgs["batch-size"] = 128 # 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, ...
cfgs['runs'] = 10
cfgs["epochs"] = 30

# -pick computation device
devcnt = torch.cuda.device_count()

# - setup multiprocessing, human signature
# random seed for python libraries
cfgs["seed"] = 2023
cfgs["var_seed"] = 1
worker_seed = cfgs["seed"] % (2**32)
cfgs["num-workers"] = 0 #3
cfgs["human_signature"] = "oas" # for signing experiments ran.


if devcnt > 1:
  cfgs["device"] = torch.device('cuda') # "cuda:1"
  torch.cuda.set_device(1)
elif devcnt == 1:
  cfgs["device"] = torch.device('cuda')
else:
  cfgs["device"] = torch.device('cpu')
#
if remotelyrun:
  cfgs["remote"] = True
else:
  cfgs["remote"] = False

##### (Generate) or Load the Dataset

In [ ]:

#----------------------
# Download/Load Dataset
#----------------------
if not cfgs["remote"]:
  # same dir. as source-code
  data_folder = "../data"
else:
  # dalton's scratch on OSU's eecs server
  data_folder = "/scratch/data"


# CIFAR10
if cfgs["dataset"] == "cifar10":
    train_data = datasets.CIFAR10(
      root=data_folder+"/CIFAR10",
      train=True,
      download=True,
      transform=transforms.Compose([ToTensor(),        
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),]),
    )
    test_data = datasets.CIFAR10(
      root=data_folder+"/CIFAR10",
      train=False,
      download= True,
      transform=transforms.Compose([ToTensor(),  
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),])
    )
    indim = [3,32,32]
    class_num = 10
    channels = indim[0]
    val_data = None
    
# Tests 
trainloader = torch.utils.data.DataLoader(train_data, batch_size=cfgs['batch-size'],shuffle=True)
# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)
pass

##### Construct Model (Differentiable Function) with  necessary methods.

In [ ]:
'''
Uses ResNetC
'''

class thisNN(nn.Module):
  def __init__(self,num_heads=1,in_feats=[3,32,32],out_feats=10,layers=6, loss_type="mce", outs_class=True, mdl_name="nnresnet") -> None:
    """
    __init__ _summary_

    _extended_summary_

    Args:
        num_heads (int, req): net. output head. Defaults to 1.
        in_feats (int, req): net. input dimension. Defaults to 1.
        out_feats (int, req): net. output dimension per head. Defaults to 1.
        loss_type (str, req): type of loss function Defaults to "mse". ("mse","mce" for multi-class cross-entropy, "bce" binary-class cross-entropy)
        outs_class (bool, req): does model return class indices. Defaults to False, implying it outputs real-valued probabilities. This is also true for loss_type="mse" regardless of user-choice.
    """
    super().__init__()
    self.num_heads = num_heads
    
    # define mdl. params.
    self.cnn = nn.ModuleList()
    for hid in range(num_heads): 
      self.cnn.append(
        ResNetC(in_chans=in_feats[0],stem_chans=64,layers=str(layers),botneck=False,num_classes=out_feats,num_heads=num_heads) 
      )
    self.relu = nn.ReLU()
    self.flatten = nn.Flatten()
    
    # prob.fcn at last output or network head layer
    if loss_type == "bce" or loss_type == "mse":
      self.out = torch.nn.Sigmoid()
    else:
      self.out = torch.nn.Softmax()
    
    self.loss_type = loss_type
    self.outs_class = outs_class
    
    # optimizer
    self.sgm = None
    
    # saved trained model
    self.chkpt = None
    
    
    self.devcnt = torch.cuda.device_count()
    self.device = None
    self.name = mdl_name
    
    if not(loss_type == "mse" or loss_type =="mce" or loss_type == "bce"):
      raise ValueError(f"loss_type='{loss_type}' is not defined for this class")
    if (loss_type == "mse"):
      self.outs_class = False    
    
@clsmethod(thisNN)
def forward(self:thisNN, x):
  # pre-process
  logits = []
  for hid in range(self.num_heads):
    resnet = self.cnn[hid]
    y = resnet(x)  
    logits.append(y)
    
  return logits # List [Tensor]

@clsmethod(thisNN)
def outhead(self:thisNN, x):
  probs = []
  for hid in range(self.num_heads):
    y = self.out(x[hid])
    probs.append(y)  
  return probs # List [Tensor]

# Add necessary functions to train model
add_to_thisnn(thisNN,worker_seed,setseed,cseedwk)

# Use saved mdl as
# mdl <-- load model function
# prediction = mdl.infer(input)

##### Run the Network Modeling (Training) 

In [ ]:
# Test.

torch.manual_seed(2023)
devcnt = torch.cuda.device_count()

cfgs['mdl_name_dir'] = "nndemo7f_cifar10_resnet6"
isclassifier = True
cfgs["ss_init"] = 1e-3 # default 1e-3
cfgs["eps_ss"] = 1 # default 5e-1
cfgs["weight_decay"] = 1e-5 # default 1e-5
mdl = thisNN(mdl_name=cfgs['mdl_name_dir'])
if devcnt > 0:
    mdl.cuda()
cfgs['pathstr'] = str(mdl.name)+str(cfgs["batch-size"])+"_"+cfgs["dataset"]+"_"+str(cfgs["runs"])+"_"+str(cfgs["epochs"])

mdl.runs(train_data, test_data, cfgs=cfgs, eval_name="Test")

In [ ]:
# Plots
plotter_v1(cfgs,run_idx=1)